In [ ]:
#!/usr/bin/env python3
"""
Combined panel figure — 3 rows
================================
Row 1 — Confusion (2 cols):
  Left col (stacked): a = coarse CM (top), c = coarse bar (bottom)
  Right col (full):   b = 21×21 fine CM

Row 2 — Performance (3 cols):
  d = combined F1 + BalAcc bar
  e = per-class F1 dumbbell
  f = per-class Accuracy dumbbell

Row 3 — Confidence (2 cols):
  g = joint confidence bucket bar chart
  h = performance vs threshold sweep

Usage: python fig_combined.py
"""

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from pathlib import Path

# ── Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "axes.titlesize":     10,
    "axes.labelsize":     10,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
})

# ── Paths ─────────────────────────────────────────────────────────────────
BASE       = Path("/hpc/group/jilab/rz179/MorphPT_MOE/experiments")
MOE_RESULT = Path(
    "/hpc/group/jilab/rz179/MorphPT_MOE/results/"
    "moe_e2e_nobreast_6clusters_filtered_v3_with_expert_margin_with_gate_2_5x"
)

MODEL_PATHS = {
    "ResNet-50":    BASE / "baseline_resnet50_freeze_2p5x"                  / "per_class.json",
    "Swin-B":       BASE / "baseline_swin_base_patch4_window7_224_2p5x"     / "per_class.json",
    "DINOv2 ViT-B": BASE / "baseline_vit_base_patch16_dinov3.lvd1689m_2p5x" / "per_class.json",
    "MorphPT":  MOE_RESULT / "per_class_soft.json",
}

MODEL_COLORS = {
    "ResNet-50":    "#E08430",   # orange — distinct from Stromal gray
    "Swin-B":       "#2CA02C",   # green
    "DINOv2 ViT-B": "#378ADD",   # blue
    "MorphPT":      "#E24B4A",   # red 
}

MOE_KEY  = "MorphPT"
BL_EDGE  = "#666666"
LINE_COL = "#BBBBBB"

MARGIN_COL = "router_x_maxpe"
PRED_COL   = "pred_soft"
N_STEPS    = 200

# Confidence colors
C_BAL = "#378ADD"
C_F1  = "#1D9E75"
C_ACC = "#E24B4A"
C_BAR = "#B5D4F4"
C_THR = "#D4700A"

# ── Taxonomy ──────────────────────────────────────────────────────────────
FINE_TO_COARSE = {
    "Colon cancer cells":        "Cancer",
    "Liver cancer cells":        "Cancer",
    "Lung cancer cells":         "Cancer",
    "Ovary cancer cells":        "Cancer",
    "Pancreas cancer cells":     "Cancer",
    "Skin cancer cells":         "Cancer",
    "B cells":                   "Lymphoid",
    "NK cells":                  "Lymphoid",
    "T cells":                   "Lymphoid",
    "Astrocytes":                "Neuroglial",
    "Microglia":                 "Neuroglial",
    "Neurons":                   "Neuroglial",
    "Oligodendrocytes":          "Neuroglial",
    "Endothelial cells":         "Tissue_Vascular",
    "Epithelial cells":          "Tissue_Vascular",
    "Fibroblasts":               "Tissue_Vascular",
    "Myeloid cells":             "Tissue_Vascular",
    "Pericytes":                 "Tissue_Vascular",
    "Smooth muscle cells":       "Tissue_Vascular",
    "Stem and progenitor cells": "Stem_Progenitor",
    "Stromal cells":             "Stromal",
}

COARSE_ORDER = [
    "Cancer", "Lymphoid", "Neuroglial",
    "Stem_Progenitor", "Stromal", "Tissue_Vascular",
]

COARSE_COLORS = {
    "Cancer":          "#C0392B",
    "Lymphoid":        "#2471A3",
    "Neuroglial":      "#1E8449",
    "Tissue_Vascular": "#CA6F1E",
    "Stem_Progenitor": "#6C3483",
    "Stromal":         "#717D7E",
}

OVERALL_ROUTER_BAL = 0.80
OVERALL_E2E_MF1    = 0.6109
OVERALL_E2E_BAL    = 0.6294


# ═══════════════════════════════════════════════════════════════════════════
# Load all data
# ═══════════════════════════════════════════════════════════════════════════
def load_per_class(path):
    rows = json.loads(Path(path).read_text())
    return {r["class"]: r for r in rows}

print("Loading performance data...")
RESULTS = {}
for model, path in MODEL_PATHS.items():
    p = Path(path)
    if not p.exists():
        print(f"  WARNING: {p} not found")
        continue
    RESULTS[model] = load_per_class(p)
    print(f"  Loaded {model}: {len(RESULTS[model])} classes")

models      = list(RESULTS.keys())
bl_models   = [m for m in models if m != MOE_KEY]
all_classes = sorted(RESULTS[models[0]].keys())

def get_metric(model, cls, metric):
    return RESULTS.get(model, {}).get(cls, {}).get(metric, 0.0)

def global_mean(model, metric):
    vals = [get_metric(model, c, metric) for c in all_classes
            if get_metric(model, c, "n") > 0]
    return float(np.mean(vals)) if vals else 0.0

def global_acc(model):
    total_n  = sum(get_metric(model, c, "n") for c in all_classes)
    total_tp = sum(
        round(get_metric(model, c, "rec") * get_metric(model, c, "n"))
        for c in all_classes
    )
    return total_tp / total_n if total_n > 0 else 0.0

print("Loading confusion data...")
cm_json     = json.loads((MOE_RESULT / "confusion_matrix_soft.json").read_text())
json_names  = cm_json["class_names"]
cm21_raw    = np.array(cm_json["matrix"], dtype=np.int64)
json_to_idx = {c: i for i, c in enumerate(json_names)}

sorted_fine = []
for coarse in COARSE_ORDER:
    sorted_fine += sorted(c for c in json_names
                          if FINE_TO_COARSE.get(c) == coarse)

sort_idx  = [json_to_idx[c] for c in sorted_fine]
cm21_sort = cm21_raw[np.ix_(sort_idx, sort_idx)]
cm21_pct  = 100 * cm21_sort / cm21_sort.sum(axis=1, keepdims=True).clip(min=1)
n_fine    = len(sorted_fine)

boundaries = []
pos = 0
for coarse in COARSE_ORDER:
    pos += sum(1 for c in sorted_fine if FINE_TO_COARSE.get(c) == coarse)
    boundaries.append(pos)

print("Loading predictions parquet...")
df        = pd.read_parquet(MOE_RESULT / "predictions.parquet")
n_coarse  = len(COARSE_ORDER)
c_to_idx  = {c: i for i, c in enumerate(COARSE_ORDER)}

cm6 = np.zeros((n_coarse, n_coarse), dtype=np.int64)
for t, p in zip(df["coarse_label"].values, df["router_pred"].values):
    ti, pi = c_to_idx.get(t, -1), c_to_idx.get(p, -1)
    if ti >= 0 and pi >= 0:
        cm6[ti, pi] += 1
cm6_pct = 100 * cm6 / cm6.sum(axis=1, keepdims=True).clip(min=1)

pc_rows    = json.loads((MOE_RESULT / "per_class_soft.json").read_text())
pc_by_name = {r["class"]: r for r in pc_rows}

coarse_rows = []
for coarse in COARSE_ORDER:
    fines   = [c for c in sorted_fine if FINE_TO_COARSE.get(c) == coarse]
    f1s     = [pc_by_name[c]["f1"]  for c in fines if c in pc_by_name]
    recs    = [pc_by_name[c]["rec"] for c in fines if c in pc_by_name]
    n_cells = int(sum(pc_by_name[c].get("n", 0) for c in fines if c in pc_by_name))
    # accuracy = weighted sum of per-class recall (tp/total_n)
    total_tp = sum(
        round(pc_by_name[c]["rec"] * pc_by_name[c]["n"])
        for c in fines if c in pc_by_name
    )
    acc_val = total_tp / n_cells if n_cells > 0 else 0.0
    coarse_rows.append({
        "coarse":   coarse,
        "n_cells":  n_cells,
        "macro_f1": round(float(np.mean(f1s)),  4) if f1s  else 0.0,
        "bal_acc":  round(float(np.mean(recs)), 4) if recs else 0.0,
        "acc":      round(acc_val, 4),
    })
df_coarse = pd.DataFrame(coarse_rows)

# Confidence metrics
all_cls_conf = sorted(df["label"].unique())
N            = len(df)

def compute_conf_metrics(y_true, y_pred):
    n2i = {n: i for i, n in enumerate(all_cls_conf)}
    ti  = np.array([n2i.get(y, -1) for y in y_true])
    pi  = np.array([n2i.get(y, -1) for y in y_pred])
    rows = []
    for i in range(len(all_cls_conf)):
        tp = int(((ti==i)&(pi==i)).sum())
        fn = int(((ti==i)&(pi!=i)).sum())
        fp = int(((ti!=i)&(pi==i)).sum())
        n  = tp + fn
        if n == 0: continue
        prec = tp / max(tp+fp, 1)
        rec  = tp / max(tp+fn, 1)
        f1   = 2*prec*rec / max(prec+rec, 1e-12)
        rows.append({"rec": rec, "f1": f1})
    bal = float(np.mean([r["rec"] for r in rows]))
    mf1 = float(np.mean([r["f1"]  for r in rows]))
    acc = float((ti==pi).sum()) / max(len(ti), 1)
    return bal, mf1, acc

base_bal, base_mf1, base_acc = compute_conf_metrics(
    df["label"].values, df[PRED_COL].values
)

# Bucket data
buckets = [(i/10, (i+1)/10) for i in range(10)]
bin_labels, cells, baccs, macros, accs = [], [], [], [], []
for lo, hi in buckets:
    mask   = (df[MARGIN_COL] >= lo) & (df[MARGIN_COL] < hi)
    subset = df[mask]
    if len(subset) == 0: continue
    bal, mf1, acc = compute_conf_metrics(
        subset["label"].values, subset[PRED_COL].values)
    bin_labels.append("{:.1f}-{:.1f}".format(lo, hi))
    cells.append(len(subset))
    baccs.append(bal); macros.append(mf1); accs.append(acc)

# Discrete threshold sweep
thresholds_disc = np.arange(0.0, 1.0, 0.1)
disc_thr, disc_bal, disc_f1, disc_acc, disc_cov = [], [], [], [], []
for t in thresholds_disc:
    subset = df[df[MARGIN_COL] >= t]
    if len(subset) < 50: break
    bal, mf1, acc = compute_conf_metrics(
        subset["label"].values, subset[PRED_COL].values)
    disc_thr.append(t); disc_bal.append(bal)
    disc_f1.append(mf1); disc_acc.append(acc)
    disc_cov.append(100 * len(subset) / N)
disc_thr = np.array(disc_thr); disc_bal = np.array(disc_bal)
disc_f1  = np.array(disc_f1);  disc_acc = np.array(disc_acc)
disc_cov = np.array(disc_cov)

print("All data loaded.")


# ═══════════════════════════════════════════════════════════════════════════
# Figure layout
# ═══════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 28))

outer = gridspec.GridSpec(
    3, 1,
    figure=fig,
    height_ratios=[1.4, 0.75, 0.7],
    hspace=0.32,
)

# ── Row 1: confusion ──────────────────────────────────────────────────────
row1 = gridspec.GridSpecFromSubplotSpec(
    1, 2,
    subplot_spec=outer[0],
    width_ratios=[1, 1.6],
    wspace=0.22,
)

# Left col: a (coarse CM) + c (coarse bar) stacked
left_gs = gridspec.GridSpecFromSubplotSpec(
    2, 1,
    subplot_spec=row1[0],
    hspace=0.52,
    height_ratios=[1.05, 0.85],
)
ax_a = fig.add_subplot(left_gs[0])   # a: coarse CM
ax_c = fig.add_subplot(left_gs[1])   # c: coarse bar

# Right col: b (fine CM, large)
ax_b = fig.add_subplot(row1[1])      # b: 21×21 fine CM

# ── Row 2: performance ────────────────────────────────────────────────────

row2 = gridspec.GridSpecFromSubplotSpec(
    1, 3,
    subplot_spec=outer[1],
    width_ratios=[1.4, 1.2, 1.2],
    wspace=0.08,
)

ax_d = fig.add_subplot(row2[0])   # d: combined bar
ax_e = fig.add_subplot(row2[1])   # e: F1 dumbbell
ax_f = fig.add_subplot(row2[2])   # f: Acc dumbbell

# ── Row 3: confidence ─────────────────────────────────────────────────────
row3 = gridspec.GridSpecFromSubplotSpec(
    1, 2,
    subplot_spec=outer[2],
    width_ratios=[1, 1.6],
    wspace=0.1,
)
ax_g = fig.add_subplot(row3[0])   # g: bucket bar
ax_h = fig.add_subplot(row3[1])   # h: threshold sweep


# ═══════════════════════════════════════════════════════════════════════════
# Panel a — 6×6 coarse routing confusion matrix
# ═══════════════════════════════════════════════════════════════════════════
im6 = ax_a.imshow(cm6_pct, cmap="Blues", vmin=0, vmax=100,
                  aspect="equal", interpolation="nearest")
cb6 = plt.colorbar(im6, ax=ax_a, fraction=0.046, pad=0.04, shrink=0.85)
cb6.set_label("% of true class", fontsize=12, labelpad=4)
cb6.ax.tick_params(labelsize=12, length=2)
cb6.outline.set_linewidth(0.5)

clabels = [c.replace("_", "\n") for c in COARSE_ORDER]
ax_a.set_xticks(range(n_coarse))
ax_a.set_yticks(range(n_coarse))
ax_a.set_xticklabels(clabels, rotation=30, ha="right", fontsize=10)
ax_a.set_yticklabels(clabels, fontsize=12)

for i in range(n_coarse):
    for j in range(n_coarse):
        v = cm6_pct[i, j]
        ax_a.text(j, i, "{:.1f}".format(v),
                  ha="center", va="center", fontsize=10,
                  color="white" if v > 55 else "#333333",
                  fontweight="bold")
for k in range(n_coarse):
    ax_a.add_patch(plt.Rectangle(
        (k-0.5, k-0.5), 1, 1,
        fill=False, edgecolor="#C0392B", linewidth=1.8, zorder=3))

for spine in ax_a.spines.values(): spine.set_visible(False)
ax_a.tick_params(length=0)
ax_a.set_xlabel("Predicted Cluster", fontsize=12, labelpad=5)
ax_a.set_ylabel("True Cluster", fontsize=12, labelpad=5)
#ax_a.set_title("Coarse routing confusion matrix (row %)\n",fontsize=10, pad=6, loc="left")


# ═══════════════════════════════════════════════════════════════════════════
# Panel b — 21×21 fine confusion matrix
# ═══════════════════════════════════════════════════════════════════════════
im21 = ax_b.imshow(cm21_pct, cmap="Blues", vmin=0, vmax=100,
                   aspect="equal", interpolation="nearest")
cb21 = plt.colorbar(im21, ax=ax_b, fraction=0.020, pad=0.02, shrink=0.82)
cb21.set_label("% of true class", fontsize=12, labelpad=4)
cb21.ax.tick_params(labelsize=12, length=2)
cb21.outline.set_linewidth(0.5)

short_labels = [c.replace(" cells", "").replace(" cancer cells", "\ncancer")
                for c in sorted_fine]
ax_b.set_xticks(range(n_fine))
ax_b.set_yticks(range(n_fine))
ax_b.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=10)
ax_b.set_yticklabels(short_labels, fontsize=10)

for i in range(n_fine):
    for j in range(n_fine):
        v = cm21_pct[i, j]
        if v > 5:
            ax_b.text(j, i, "{:.0f}".format(v),
                      ha="center", va="center", fontsize=10,
                      color="white" if v > 55 else "#333333",
                      fontweight="bold")

for b in boundaries[:-1]:
    ax_b.axhline(b-0.5, color="#C0392B", linewidth=1.0, alpha=0.6)
    ax_b.axvline(b-0.5, color="#C0392B", linewidth=1.0, alpha=0.6)

for spine in ax_b.spines.values(): spine.set_visible(False)
ax_b.tick_params(length=0)
ax_b.set_xlabel("Predicted class", fontsize=12, labelpad=6)
ax_b.set_ylabel("True class", fontsize=12, labelpad=6)
#ax_b.set_title("Fine-class confusion matrix (row %)", fontsize=10,pad=8, loc="left")


# ═══════════════════════════════════════════════════════════════════════════
# Panel c — Coarse-group bar chart
# ═══════════════════════════════════════════════════════════════════════════
x_pos  = np.arange(n_coarse)
width  = 0.35
colors = [COARSE_COLORS[c] for c in COARSE_ORDER]

overall_coarse_f1  = global_mean(MOE_KEY, "f1")
overall_coarse_acc = global_acc(MOE_KEY)

#bars_f1  = ax_c.bar(x_pos - width/2, df_coarse["macro_f1"], width,color=colors, alpha=0.90, label="Macro F1", linewidth=0)
#bars_bal = ax_c.bar(x_pos + width/2, df_coarse["acc"], width,color=colors, alpha=0.40, label="Accuracy", linewidth=0)

bars_f1  = ax_c.bar(x_pos - width/2, df_coarse["macro_f1"], width,
                    color=colors, alpha=0.85, linewidth=0.8,
                    hatch="///", edgecolor="white", label="Macro F1")
bars_bal = ax_c.bar(x_pos + width/2, df_coarse["acc"], width,
                    color=colors,
                    alpha=1.0, linewidth=0, label="Accuracy")

for bar in list(bars_f1) + list(bars_bal):
    h = bar.get_height()
    ax_c.text(bar.get_x() + bar.get_width()/2, h + 0.013,
              "{:.3f}".format(h),
              ha="center", va="bottom", fontsize=12, color="#333333")

ax_c.axhline(overall_coarse_f1, color="#555555", linestyle="--",
             linewidth=0.9, alpha=0.7)
ax_c.axhline(overall_coarse_acc, color="#555555", linestyle=":",
             linewidth=0.9, alpha=0.7)

bar_xlabels = [
    "{}\n(n={:,})".format(r["coarse"].replace("_", "\n"), r["n_cells"])
    for _, r in df_coarse.iterrows()
]
ax_c.set_xticks(x_pos)
ax_c.set_xticklabels(bar_xlabels, fontsize=12)
for tick, coarse in zip(ax_c.get_xticklabels(), COARSE_ORDER):
    tick.set_color(COARSE_COLORS[coarse])

# put F1 label above, Acc label below its line
ax_c.text(n_coarse - 0.5, overall_coarse_f1 + 0.015, 
          "Overall F1 = {:.3f}".format(overall_coarse_f1),
          ha="right", va="bottom", fontsize=8, color="#333333")

ax_c.text(n_coarse - 0.5, overall_coarse_acc - 0.015,
          "Overall Acc = {:.3f}".format(overall_coarse_acc),
          ha="right", va="top", fontsize=8, color="#333333")

ax_c.set_ylim(0, 1.12)
ax_c.set_ylabel("Score", fontsize=12)
#ax_c.set_title("End-to-end performance by coarse group", fontsize=10, pad=6, loc="left")
ax_c.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax_c.grid(axis="y", color="#DDDDDD", linewidth=0.6, zorder=0)
ax_c.set_axisbelow(True)
#ax_c.legend(fontsize=12, loc="upper right", frameon=False, handlelength=1.4)
ax_c.legend(handles=[
    mpatches.Patch(facecolor="#888888", hatch="///", edgecolor="white", label="Macro F1"),
    mpatches.Patch(facecolor="#888888", label="Accuracy"),
], fontsize=12, loc="upper right", frameon=False, handlelength=1.4)


# ═══════════════════════════════════════════════════════════════════════════
# Panel d — Combined F1 + Acc bar
# ═══════════════════════════════════════════════════════════════════════════
f1_vals  = [global_mean(m, "f1") for m in models]
acc_vals = [global_acc(m)        for m in models]
xs = np.arange(len(models))
bw = 0.32

for i, m in enumerate(models):
    color = MODEL_COLORS[m]
    #ax_d.bar(xs[i] - bw/2, f1_vals[i], bw,
             #color=color, alpha=1.0 if m == MOE_KEY else 0.80,
            # edgecolor="white", linewidth=0.5)
    #ax_d.bar(xs[i] + bw/2, acc_vals[i], bw,
             #color=color, alpha=0.45 if m == MOE_KEY else 0.35,
             #edgecolor="white", linewidth=0.5)


    ax_d.bar(xs[i] - bw/2, f1_vals[i], bw,
         color=color, alpha=0.85, linewidth=0.8,
         hatch="///", edgecolor="white")
    ax_d.bar(xs[i] + bw/2, acc_vals[i], bw,
         color=color, alpha=1.0, linewidth=0)
    ax_d.text(xs[i] - bw/2, f1_vals[i] + 0.005,
              "{:.3f}".format(f1_vals[i]),
              ha="center", va="bottom", fontsize=12,
              color=color if m == MOE_KEY else "#222222",
              fontweight="bold" if m == MOE_KEY else "normal")
    ax_d.text(xs[i] + bw/2, acc_vals[i] + (0.03 if m == MOE_KEY else 0.005),
              "{:.3f}".format(acc_vals[i]),
              ha="center", va="bottom", fontsize=12,
              color=color if m == MOE_KEY else "#222222",
              fontweight="bold" if m == MOE_KEY else "normal")

moe_i    = models.index(MOE_KEY)
best_f1  = max(global_mean(m, "f1") for m in bl_models)
best_acc = max(global_acc(m)        for m in bl_models)
moe_f1   = global_mean(MOE_KEY, "f1")
moe_acc  = global_acc(MOE_KEY)

# +Δ labels to the right of MorphPT bars
#ax_d.text(moe_i + bw + 0.10, moe_f1, "{:+.3f}".format(moe_f1 - best_f1), va="center", ha="left", fontsize=8.5, color="#E24B4A", fontweight="bold")
#ax_d.text(moe_i + bw + 0.10, moe_acc, "{:+.3f}".format(moe_acc - best_acc), va="center", ha="left", fontsize=8.5, color="#E24B4A", fontweight="bold")

ax_d.set_xticks(xs)
ax_d.set_xticklabels(models, fontsize=12, rotation=12, ha="right")
ax_d.get_xticklabels()[moe_i].set_color("#E24B4A")
ax_d.get_xticklabels()[moe_i].set_fontweight("bold")
y_floor = min(f1_vals + acc_vals) - 0.06
ax_d.set_ylim(max(0, y_floor), min(1.0, max(f1_vals + acc_vals) + 0.12))
ax_d.set_xlim(-0.6, len(models) + 0.2)
ax_d.set_ylabel("Score", fontsize=12)
#ax_d.set_title("Overall performance by fine classes", fontsize=10, pad=8, loc="left")
ax_d.grid(axis="y", color="#EEEEEE", linewidth=0.6, zorder=0)
ax_d.set_axisbelow(True)
#ax_d.legend(handles=[
#    mpatches.Patch(facecolor="#888888", alpha=1.0,  label="Macro F1"),
#    mpatches.Patch(facecolor="#888888", alpha=0.40, label="Accuracy"),
#], fontsize=12, loc="upper left", frameon=False)

ax_d.legend(handles=[
    mpatches.Patch(facecolor="#888888", hatch="///", edgecolor="white", label="Macro F1"),
    mpatches.Patch(facecolor="#888888", label="Accuracy"),
], fontsize=12, loc="upper left", frameon=False)
# ═══════════════════════════════════════════════════════════════════════════
# Dumbbell helper
# ═══════════════════════════════════════════════════════════════════════════
def draw_dumbbell(ax, metric, xlabel, title, hide_ylabels=False):
    cls_data = []
    for cls in all_classes:
        moe_val  = get_metric(MOE_KEY, cls, metric)
        best_val = max(get_metric(m, cls, metric) for m in bl_models)
        cls_data.append({
            "cls": cls, "moe": moe_val,
            "best_bl": best_val, "delta": moe_val - best_val,
        })
    cls_data.sort(key=lambda d: d["moe"])
    nc = len(cls_data)

    for i, d in enumerate(cls_data):
        lo = min(d["best_bl"], d["moe"])
        hi = max(d["best_bl"], d["moe"])
        ax.plot([lo, hi], [i, i], color=LINE_COL,
                linewidth=1.6, alpha=0.85, zorder=1,
                solid_capstyle="round")
        ax.scatter(d["best_bl"], i, s=50, zorder=3,
                   facecolors="white", edgecolors=BL_EDGE, linewidths=1.4)
        ax.scatter(d["moe"], i, s=65, zorder=4,
                   facecolors=MODEL_COLORS[MOE_KEY],
                   edgecolors=MODEL_COLORS[MOE_KEY])
        sign = "+" if d["delta"] >= 0 else ""
        ax.text(d["moe"] + 0.014, i,
                "{}{:.3f}".format(sign, d["delta"]),
                va="center", ha="left", fontsize=12,
                color="#E24B4A" if d["delta"] >= 0 else "#222222",
                fontweight="normal")

    if hide_ylabels:
        ax.set_yticks(range(nc))
        ax.set_yticklabels([])
    else:
        ax.set_yticks(range(nc))
        ax.set_yticklabels([d["cls"] for d in cls_data],
                           fontsize=12, color="black")
    ax.invert_yaxis()

    n_pos  = sum(1 for d in cls_data if d["delta"] > 0)
    mean_d = np.mean([d["delta"] for d in cls_data])
    ax.set_xlabel(
        "{}\nWins {}/{} classes".format(
            xlabel, n_pos, nc),
        fontsize=12)
    ax.set_xlim(0, 1.14)
    #ax.set_title(title, fontsize=10, pad=8, loc="left")
    ax.grid(axis="x", color="#EEEEEE", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.legend(handles=[
        mlines.Line2D([0], [0], marker="o", color="w",
                      markerfacecolor="white", markeredgecolor=BL_EDGE,
                      markersize=8, markeredgewidth=1.4,
                      label="Best baseline"),
        mlines.Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=MODEL_COLORS[MOE_KEY],
                      markeredgecolor=MODEL_COLORS[MOE_KEY],
                      markersize=8, label="MorphPT"),
    ], fontsize=12, loc="upper right", frameon=False)


# ═══════════════════════════════════════════════════════════════════════════
# Panel e — F1 dumbbell
# ═══════════════════════════════════════════════════════════════════════════
draw_dumbbell(ax_e, "f1",
              xlabel="Macro F1  (○ best baseline  ● MorphPT)",
              title="Per-class F1")

# ═══════════════════════════════════════════════════════════════════════════
# Panel f — Accuracy dumbbell
# ═══════════════════════════════════════════════════════════════════════════
draw_dumbbell(ax_f, "rec",
              xlabel="Per-class accuracy  (○ best baseline  ● MorphPT)",
              title="Per-class accuracy",
              hide_ylabels=True)


# ═══════════════════════════════════════════════════════════════════════════
# Panel g — Joint confidence bucket bar chart
# ═══════════════════════════════════════════════════════════════════════════
ax_g2 = ax_g.twinx()

# Performance lines on LEFT axis (ax_g)
#ax_g.plot(range(len(bin_labels)), baccs, color=C_BAL, marker="o", markersize=4, linewidth=1.5,label="Balanced Acc")
ax_g.plot(range(len(bin_labels)), macros,
          color=C_F1, marker="^", markersize=4, linewidth=1.5,
          linestyle="-.", label="Macro F1")
ax_g.plot(range(len(bin_labels)), accs,
          color=C_ACC, marker="s", markersize=4, linewidth=1.5,
          linestyle="--", label="Accuracy")

#ax_g.axhline(base_bal, color=C_BAL, linestyle=":", linewidth=0.8, alpha=0.5)
ax_g.axhline(base_mf1, color=C_F1,  linestyle=":", linewidth=0.8, alpha=0.5)
ax_g.axhline(base_acc, color=C_ACC, linestyle=":", linewidth=0.8, alpha=0.5)

# Cell count bars on RIGHT axis (ax_g2)
ax_g2.bar(range(len(bin_labels)), cells,
          color=C_BAR, alpha=0.35, zorder=1)

ax_g.set_xticks(range(len(bin_labels)))
ax_g.set_xticklabels(bin_labels, rotation=45, ha="right", fontsize=10)
ax_g.set_xlabel("Joint confidence bucket  (Mr × max Pe)", fontsize=12)
ax_g.set_ylabel("Performance", fontsize=12)
ax_g2.set_ylabel("Cell count", fontsize=12)
ax_g.set_ylim(0.28, 1.05)
ax_g.tick_params(axis="y", labelsize=12)
ax_g2.tick_params(labelsize=12)
ax_g.grid(axis="y", linestyle=":", alpha=0.3, zorder=0)
#ax_g.set_title("Performance by confidence bucket", fontsize=10,pad=8, loc="left")

legend_handles = [
    Line2D([0],[0], color=C_BAR, linewidth=6, alpha=0.45, label="Cell count"),
    #Line2D([0],[0], color=C_BAL, linewidth=1.5, marker="o", markersize=4, label="Balanced Acc"),
    Line2D([0],[0], color=C_F1,  linewidth=1.5, marker="^", markersize=4,
           linestyle="-.", label="Macro F1"),
    Line2D([0],[0], color=C_ACC, linewidth=1.5, marker="s", markersize=4,
           linestyle="--", label="Accuracy"),
]
ax_g.legend(handles=legend_handles, fontsize=12, loc="upper left",
            framealpha=0.85, handlelength=1.5)


# ═══════════════════════════════════════════════════════════════════════════
# Panel h — Performance vs threshold sweep
# ═══════════════════════════════════════════════════════════════════════════
#ax_h.plot(disc_thr, disc_bal, color=C_BAL, marker="o", markersize=5, linewidth=2.0, label="Balanced Acc")
ax_h.plot(disc_thr, disc_f1,  color=C_F1,  marker="^", markersize=5,
          linewidth=2.0, linestyle="-.", label="Macro F1")
ax_h.plot(disc_thr, disc_acc, color=C_ACC, marker="s", markersize=5,
          linewidth=2.0, linestyle="--", label="Accuracy")

for t, b, c in zip(disc_thr, disc_acc, disc_cov):
    ax_h.annotate("{:.3f}\n({:.0f}%)".format(b, c),
                  (t, b), textcoords="offset points",
                  xytext=(0, 9), ha="center", fontsize=12,
                  color=C_BAL)

#ax_h.axhline(base_bal, color=C_BAL, linestyle=":", linewidth=1.0, alpha=0.5, label="Baseline Bal Acc = {:.3f}".format(base_bal))
ax_h.axhline(base_mf1, color=C_F1,  linestyle=":", linewidth=1.0, alpha=0.5,
             label="Baseline Macro F1 = {:.3f}".format(base_mf1))
ax_h.axhline(base_acc, color=C_ACC, linestyle=":", linewidth=1.0, alpha=0.5,
             label="Baseline Acc = {:.3f}".format(base_acc))

ax_h.axvline(0.5, color=C_THR, linestyle=":", linewidth=1.2, alpha=0.8)
ax_h.text(0.51, base_bal - 0.04, "C≥0.5", fontsize=12,
          color=C_THR, style="italic")

ax_h.set_xlabel(
    "Joint confidence threshold  (C = Mr × max Pe ≥ threshold)",
    fontsize=12)
# y-label shared with panel g — omit to avoid repetition
ax_h.set_xticks(disc_thr)
ax_h.set_xticklabels(["≥{:.1f}".format(t) for t in disc_thr], fontsize=12)
ax_h.set_ylim(base_acc - 0.06, 1.0)
ax_h.tick_params(labelsize=12)
ax_h.grid(linestyle=":", alpha=0.3)
ax_h.legend(fontsize=12, loc="upper left", framealpha=0.85,
            ncol=2, handlelength=1.5)
#ax_h.set_title("Performance vs confidence threshold", fontsize=10,pad=8, loc="left")

# Secondary x-axis: coverage %
ax_h2 = ax_h.twiny()
ax_h2.set_xlim(ax_h.get_xlim())
ax_h2.set_xticks(disc_thr)
ax_h2.set_xticklabels(["{:.0f}%".format(c) for c in disc_cov], fontsize=12)
ax_h2.set_xlabel("Coverage (% cells retained)", fontsize=12, labelpad=3)

fig.canvas.draw()

pos = ax_d.get_position()
new_w = pos.width  * 0.75   # 宽度缩到 70%
new_h = pos.height   
new_y = pos.y0 + (pos.height - new_h) / 2   # 垂直居中
ax_d.set_position([pos.x0, new_y, new_w, new_h])


pos = ax_g.get_position()
new_w = pos.width  * 0.90
new_h = pos.height * 0.88
new_x = pos.x0 + (pos.width  - new_w) / 2   # keep horizontally centered
new_y = pos.y0 + (pos.height - new_h) / 2   # keep vertically centered
ax_g.set_position([new_x, new_y, new_w, new_h])
ax_g2.set_position([new_x, new_y, new_w, new_h])

pos = ax_h.get_position()
new_w = pos.width  * 0.90
new_h = pos.height * 0.90
new_x = pos.x0 + (pos.width  - new_w) / 2   # keep horizontally centered
new_y = pos.y0 + (pos.height - new_h) / 2   # keep vertically centered
ax_h.set_position([new_x, new_y, new_w, new_h])
ax_h2.set_position(ax_h.get_position())


panel_axes = [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g, ax_h]
panel_lbls = ["a",  "b",  "c",  "d",  "e",  "f",  "g",  "h"]
for ax, lbl in zip(panel_axes, panel_lbls):
    cell_bbox = ax.get_subplotspec().get_position(fig)  # grid cell in figure coords
    fig.text(cell_bbox.x0, cell_bbox.y1, lbl,
             fontsize=20, fontweight="bold",
             ha="left", va="bottom",
             transform=fig.transFigure)

# ── Save ──────────────────────────────────────────────────────────────────
for ext in ("pdf", "png"):
    dpi = 300 if ext == "pdf" else 150
    fig.savefig("fig_combined.{}".format(ext), format=ext,
                dpi=dpi, bbox_inches="tight")
    print("Saved: fig_combined.{}".format(ext))

plt.show()
print("Done.")